In [5]:
# ── Quantitative Evaluation Cell ─────────────────────────────────────────────
# Compares prediction .txt vs ground truth .txt and produces:
#   - Confusion matrix
#   - Per-class: Precision, Recall, F1, IoU
#   - Overall Accuracy, mIoU
#
# Assumes both files have same number of points (same order).
# If prediction has fewer points, uses KDTree to match to GT.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
from pathlib import Path
from scipy.spatial import KDTree

# ── Parameters ────────────────────────────────────────────────────────────
PRED_FILE         = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/Dataset/Train/final_20230918_M1_I74_50-55_5cmds_9_shifted.txt")
GT_FILE           = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/Dataset/Train/final_20230918_M1_I74_50-55_5cmds_9_shifted.txt")
OUTPUT_DIR        = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/results")

NUM_CLASSES       = 10
PRED_LABEL_COL    = -1    # last column of prediction file = class label
GT_LABEL_COL      = -1    # last column of GT file = class label
PRED_XYZ_COLS     = [0, 1, 2]
GT_XYZ_COLS       = [0, 1, 2]
USE_KDTREE        = True   # True  = match by nearest XYZ (handles different point counts)
                            # False = match by row index (files must have same rows)
DISTANCE_THRESHOLD = 0.1   # max distance for KDTree match (metres)

CLASS_NAMES = [
    "Unlabeled",                     # class 0
    "Bridge-Deck,Beam&Girder",       # class 1
    "Bridge-Abutment&Wing wall",     # class 2
    "Bridge-Pier",                   # class 3
    "Man-made terrain",              # class 4
    "Natural terrain",               # class 5
    "Vegetation",                    # class 6
    "Buildings",                     # class 7
    "Remaining hardscape",           # class 8
    "Scanning artifacts",            # class 9
]
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load files ────────────────────────────────────────────────────────────
def load_file(path, xyz_cols, label_col):
    pts, labels = [], []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            xyz   = [float(parts[c]) for c in xyz_cols]
            label = int(float(parts[label_col]))
            pts.append(xyz)
            labels.append(label)
    return np.array(pts, dtype=np.float32), np.array(labels, dtype=np.int32)

print("Loading prediction file ...")
pred_xyz, pred_labels = load_file(PRED_FILE, PRED_XYZ_COLS, PRED_LABEL_COL)
print(f"  Points : {len(pred_labels):,}  |  Labels: {np.unique(pred_labels).tolist()}")

print("Loading ground truth file ...")
gt_xyz, gt_labels = load_file(GT_FILE, GT_XYZ_COLS, GT_LABEL_COL)
print(f"  Points : {len(gt_labels):,}  |  Labels: {np.unique(gt_labels).tolist()}")

# ── Build confusion matrix ─────────────────────────────────────────────────
print("\nBuilding confusion matrix ...")
CM = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

if USE_KDTREE or len(pred_labels) != len(gt_labels):
    print("  Using KDTree matching (pred → GT nearest neighbor) ...")
    tree = KDTree(gt_xyz)
    for i, (pt, pl) in enumerate(zip(pred_xyz, pred_labels)):
        dist, idx = tree.query(pt)
        if dist <= DISTANCE_THRESHOLD:
            gt_l = gt_labels[idx]
            r = np.clip(pl,  0, NUM_CLASSES - 1)
            c = np.clip(gt_l, 0, NUM_CLASSES - 1)
            CM[r, c] += 1
        if i % 50000 == 0:
            print(f"  Processed {i:,} / {len(pred_labels):,} ...", end="\r")
else:
    print("  Using direct row matching ...")
    for pl, gl in zip(pred_labels, gt_labels):
        r = np.clip(pl, 0, NUM_CLASSES - 1)
        c = np.clip(gl, 0, NUM_CLASSES - 1)
        CM[r, c] += 1

print(f"\nConfusion matrix built. Total matched points: {CM.sum():,}")

# ── Per-class metrics ──────────────────────────────────────────────────────
TP        = np.array([CM[i, i]            for i in range(NUM_CLASSES)], dtype=np.float64)
FP        = np.array([CM[i, :].sum() - CM[i, i] for i in range(NUM_CLASSES)], dtype=np.float64)
FN        = np.array([CM[:, i].sum() - CM[i, i] for i in range(NUM_CLASSES)], dtype=np.float64)

precision = np.where((TP + FP) > 0, TP / (TP + FP), 0.0)
recall    = np.where((TP + FN) > 0, TP / (TP + FN), 0.0)
f1        = np.where((precision + recall) > 0,
                     2 * precision * recall / (precision + recall), 0.0)
iou       = np.where((TP + FP + FN) > 0, TP / (TP + FP + FN), 0.0)

overall_acc = TP.sum() / CM.sum() if CM.sum() > 0 else 0.0
mean_iou    = np.nanmean(iou[iou > 0])   # ignore empty classes
mean_f1     = np.nanmean(f1[f1 > 0])

# ── Print table ───────────────────────────────────────────────────────────
print("\n" + "=" * 85)
print(f"{'Class':<22} {'Points':>8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'IoU':>8}")
print("=" * 85)
for i in range(NUM_CLASSES):
    n_pts = int(CM[:, i].sum())   # GT points in this class
    name  = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"Class {i}"
    empty = "(empty)" if n_pts == 0 else ""
    print(f"  {name:<20} {n_pts:>8,} {precision[i]:>10.3f} {recall[i]:>8.3f} "
          f"{f1[i]:>8.3f} {iou[i]:>8.3f}  {empty}")
print("=" * 85)
print(f"  {'Overall Accuracy':<20} {CM.sum():>8,} {'':>10} {'':>8} {'':>8} {overall_acc:>8.3f}")
print(f"  {'Mean IoU':<20} {'':>8} {'':>10} {'':>8} {'':>8} {mean_iou:>8.3f}")
print(f"  {'Mean F1':<20} {'':>8} {'':>10} {'':>8} {mean_f1:>8.3f}")
print("=" * 85)

# ── Save confusion matrix ─────────────────────────────────────────────────
cm_path = OUTPUT_DIR / "confusion_matrix.txt"
np.savetxt(str(cm_path), CM, fmt="%d")
print(f"\nConfusion matrix saved → '{cm_path}'")

# ── Save full accuracy report ─────────────────────────────────────────────
report_path = OUTPUT_DIR / "accuracy_report.txt"
with open(report_path, "w") as f:
    f.write("=" * 85 + "\n")
    f.write(f"{'Class':<22} {'Points':>8} {'Precision':>10} {'Recall':>8} "
            f"{'F1':>8} {'IoU':>8}\n")
    f.write("=" * 85 + "\n")
    for i in range(NUM_CLASSES):
        n_pts = int(CM[:, i].sum())
        name  = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"Class {i}"
        f.write(f"  {name:<20} {n_pts:>8,} {precision[i]:>10.3f} {recall[i]:>8.3f} "
                f"{f1[i]:>8.3f} {iou[i]:>8.3f}\n")
    f.write("=" * 85 + "\n")
    f.write(f"  {'Overall Accuracy':<20} {CM.sum():>8,} {'':>10} {'':>8} "
            f"{'':>8} {overall_acc:>8.3f}\n")
    f.write(f"  {'Mean IoU (non-empty)':<20} {'':>8} {'':>10} {'':>8} "
            f"{'':>8} {mean_iou:>8.3f}\n")
    f.write(f"  {'Mean F1 (non-empty)':<20} {'':>8} {'':>10} "
            f"{'':>8} {mean_f1:>8.3f}\n")
    f.write("=" * 85 + "\n")
    f.write("\nConfusion Matrix (rows=pred, cols=GT):\n")
    f.write(f"{'':>22}")
    for name in CLASS_NAMES:
        f.write(f"{name[:8]:>10}")
    f.write("\n")
    for i in range(NUM_CLASSES):
        name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"Class {i}"
        f.write(f"  {name:<20}")
        for j in range(NUM_CLASSES):
            f.write(f"{CM[i,j]:>10,}")
        f.write("\n")

print(f"Accuracy report saved  → '{report_path}'")

Loading prediction file ...
  Points : 2,147,469  |  Labels: [1, 3, 4, 5, 6, 7, 9, 10]
Loading ground truth file ...
  Points : 2,147,469  |  Labels: [1, 3, 4, 5, 6, 7, 9, 10]

Building confusion matrix ...
  Using KDTree matching (pred → GT nearest neighbor) ...
  Processed 2,100,000 / 2,147,469 ...
Confusion matrix built. Total matched points: 2,147,469

Class                    Points  Precision   Recall       F1      IoU
  Unlabeled                   0      0.000    0.000    0.000    0.000  (empty)
  Bridge-Deck,Beam&Girder      125      1.000    1.000    1.000    1.000  
  Bridge-Abutment&Wing wall        0      0.000    0.000    0.000    0.000  (empty)
  Bridge-Pier             1,740      1.000    1.000    1.000    1.000  
  Man-made terrain        1,745      1.000    1.000    1.000    1.000  
  Natural terrain       440,621      1.000    1.000    1.000    1.000  
  Vegetation            986,414      1.000    1.000    1.000    1.000  
  Buildings             701,226      1.000   

/tmp/ipykernel_392842/1641333160.py:97: RuntimeWarning: invalid value encountered in divide
  precision = np.where((TP + FP) > 0, TP / (TP + FP), 0.0)
/tmp/ipykernel_392842/1641333160.py:98: RuntimeWarning: invalid value encountered in divide
  recall    = np.where((TP + FN) > 0, TP / (TP + FN), 0.0)
/tmp/ipykernel_392842/1641333160.py:100: RuntimeWarning: invalid value encountered in divide
  2 * precision * recall / (precision + recall), 0.0)
/tmp/ipykernel_392842/1641333160.py:101: RuntimeWarning: invalid value encountered in divide
  iou       = np.where((TP + FP + FN) > 0, TP / (TP + FP + FN), 0.0)
